# 04 — Evaluation

Metrics against ground truth with mir_eval, plus error analysis: which notes are missed or hallucinated.

In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "tests").exists():
    ROOT = ROOT.parent
AUDIO_DIR = ROOT / "tests" / "fixtures" / "audio"
GOLDEN_DIR = ROOT / "tests" / "fixtures" / "golden"
print("project root:", ROOT)


project root: /home/anton/Projects/music-teacher


In [2]:
import json
import numpy as np
from music_teacher.evaluation.fixtures import default_fixtures, write_fixture
from music_teacher.evaluation.metrics import score
from music_teacher.transcription.audio import load_audio
from music_teacher.transcription.onnx_engine import BasicPitchEngine

for fixture in default_fixtures():
    write_fixture(fixture, AUDIO_DIR)
engine = BasicPitchEngine(device="auto"); engine.warmup()


In [3]:
def reference(name):
    data = json.loads((AUDIO_DIR / f"{name}.notes.json").read_text())
    intervals = np.array([[n["start"], n["end"]] for n in data["notes"]], dtype=float)
    pitches = np.array([n["pitch"] for n in data["notes"]], dtype=float)
    return intervals, pitches

rows = {}
for fixture in default_fixtures():
    ref_i, ref_p = reference(fixture.name)
    events = engine.transcribe(AUDIO_DIR / f"{fixture.name}.wav")
    est_i = np.array([[e.start, e.end] for e in events], dtype=float) if events else np.zeros((0, 2))
    est_p = np.array([e.pitch for e in events], dtype=float)
    metrics = score(ref_i, ref_p, est_i, est_p)
    rows[fixture.name] = metrics
    print(f"{fixture.name:<16} P={metrics['onset_pitch_precision']:.3f} R={metrics['onset_pitch_recall']:.3f} F1={metrics['onset_pitch_f1']:.3f}")
print("mean F1:", np.mean([m['onset_pitch_f1'] for m in rows.values()]))


sine_a4          P=1.000 R=1.000 F1=1.000
sine_scale       P=1.000 R=1.000 F1=1.000
flute_melody     P=1.000 R=1.000 F1=1.000
flute_noisy      P=1.000 R=1.000 F1=1.000
piano_chords     P=1.000 R=1.000 F1=1.000
mean F1: 1.0


In [4]:
# error analysis for one fixture
name = "flute_melody"
ref_i, ref_p = reference(name)
events = engine.transcribe(AUDIO_DIR / f"{name}.wav")
est_p = [e.pitch for e in events]
print("reference:", sorted(ref_p.astype(int).tolist()))
print("estimated:", sorted(est_p))
print("extra    :", sorted(set(est_p) - set(ref_p.astype(int).tolist())))
print("missing  :", sorted(set(ref_p.astype(int).tolist()) - set(est_p)))


reference: [72, 72, 74, 76, 76, 79, 79, 81]
estimated: [72, 72, 74, 76, 76, 79, 79, 81]
extra    : []
missing  : []


**Workflow:** change a post-processing rule, run `uv run python scripts/evaluate.py`, and only update `tests/fixtures/baseline.json` when the change is a real improvement.